# Feature Engineering for PTM Prediction

This notebook extracts comprehensive features from protein sequences for Random Forest and XGBoost models.

## Overview
- **Input**: Raw protein sequences (31 amino acids)
- **Output**: Numerical feature matrix for ML models
- **Memory**: Optimized for large datasets (89K+ sequences)

## Features Extracted:
1. **Amino Acid Composition** (20 features) - % of each AA
2. **Cysteine Features** (8 features) - Position, count, context
3. **Physicochemical Properties** (4 features) - Hydrophobicity, charge, etc.
4. **Position Features** (1 feature) - Middle amino acid
5. **Complexity** (1 feature) - Unique AA count

**Total**: ~35 base features → ~100+ after encoding

## Configuration

**Modify these parameters as needed:**

In [34]:
# ============================================================================
# CONFIGURATION PARAMETERS - MODIFY THESE AS NEEDED
# ============================================================================

# File paths
#INPUT_FILE = '../data/train.csv'              # Input CSV file
#INPUT_FILE = '../data/train_augmented_v2.csv'              # Input CSV file
INPUT_FILE = '../data/independent_test_input.csv'              # Input CSV file
#OUTPUT_FILE = '../data_engineered/train_with_features.csv'  # Output file with features
#OUTPUT_FILE = '../data_engineered/train_augmented_v2_with_features.csv'  # Output file with features
OUTPUT_FILE = '../data_engineered/independent_test_input_with_features.csv'  # Output file with features

# Processing parameters
CHUNK_SIZE = 5000                             # Number of sequences per chunk (reduce if memory issues)
MODE = 'lightweight'                          # 'lightweight' or 'minimal'

# Feature extraction mode
# - 'lightweight': Essential features only (~35 features)
# - 'minimal': Only AA composition + cysteine count (21 features)

print("Configuration loaded:")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Mode:   {MODE}")
print(f"  Chunk size: {CHUNK_SIZE:,}")

Configuration loaded:
  Input:  ../data/independent_test_input.csv
  Output: ../data_engineered/independent_test_input_with_features.csv
  Mode:   lightweight
  Chunk size: 5,000


## 1. Import Libraries

In [35]:
import pandas as pd
import numpy as np
from collections import Counter
import gc  # Garbage collection for memory management
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 2. Define Physicochemical Properties

These dictionaries contain scientifically measured properties for each amino acid.

In [36]:
# Hydrophobicity scale (Kyte-Doolittle)
# Positive = hydrophobic, Negative = hydrophilic
HYDROPHOBICITY = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Molecular weight (Daltons)
MOLECULAR_WEIGHT = {
    'A': 89.09, 'R': 174.20, 'N': 132.12, 'D': 133.10, 'C': 121.15,
    'Q': 146.15, 'E': 147.13, 'G': 75.07, 'H': 155.16, 'I': 131.17,
    'L': 131.17, 'K': 146.19, 'M': 149.21, 'F': 165.19, 'P': 115.13,
    'S': 105.09, 'T': 119.12, 'W': 204.23, 'Y': 181.19, 'V': 117.15
}

# Charge at pH 7
CHARGE = {
    'A': 0, 'R': 1, 'N': 0, 'D': -1, 'C': 0,
    'Q': 0, 'E': -1, 'G': 0, 'H': 0.1, 'I': 0,
    'L': 0, 'K': 1, 'M': 0, 'F': 0, 'P': 0,
    'S': 0, 'T': 0, 'W': 0, 'Y': 0, 'V': 0
}

# Standard 20 amino acids
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

print("✓ Physicochemical properties defined")

✓ Physicochemical properties defined


## 3. Feature Extraction Functions

### 3.1 Amino Acid Composition

In [37]:
def get_aa_composition(sequence):
    """
    Calculate amino acid composition.
    
    Returns: Dictionary with % of each amino acid
    Example: {'aa_A': 0.1, 'aa_C': 0.05, ...}
    """
    counter = Counter(sequence)
    total = len(sequence)
    features = {}
    
    for aa in AMINO_ACIDS:
        features[f'aa_{aa}'] = counter.get(aa, 0) / total
    
    return features

# Test the function
test_seq = "ACDEFGHIKLMNPQRSTVWYACDEFGHIKL"
test_features = get_aa_composition(test_seq)
print(f"✓ AA composition function defined")
print(f"  Example: {list(test_features.items())[:3]}")

✓ AA composition function defined
  Example: [('aa_A', 0.06666666666666667), ('aa_C', 0.06666666666666667), ('aa_D', 0.06666666666666667)]


### 3.2 Cysteine-Specific Features

**Critical for PTM prediction!** All three target modifications are cysteine-related.

In [38]:
def get_cysteine_features(sequence):
    """
    Extract cysteine-specific features.
    
    Features:
    - cys_count: Number of cysteines
    - cys_present: 1 if has cysteine, 0 otherwise
    - cys_first_pos: Position of first cysteine
    - cys_middle_dist: Distance of nearest cysteine from middle (position 15)
    - cys_ctx_{-2,-1,1,2}: Amino acids around first cysteine
    """
    features = {}
    
    # Find all cysteine positions
    cys_positions = [i for i, aa in enumerate(sequence) if aa == 'C']
    
    # Basic counts
    features['cys_count'] = len(cys_positions)
    features['cys_present'] = 1 if len(cys_positions) > 0 else 0
    
    if len(cys_positions) > 0:
        # Position features
        features['cys_first_pos'] = cys_positions[0]
        
        # Distance from middle (position 15 in 31-AA sequence)
        middle_dist = min([abs(pos - 15) for pos in cys_positions])
        features['cys_middle_dist'] = middle_dist
        
        # Context around first cysteine (±2 positions)
        # Store as strings - will be one-hot encoded later
        first_cys = cys_positions[0]
        for offset in [-2, -1, 1, 2]:
            pos = first_cys + offset
            if 0 <= pos < len(sequence):
                features[f'cys_ctx_{offset}'] = sequence[pos]
            else:
                features[f'cys_ctx_{offset}'] = 'X'  # Boundary marker
    else:
        # No cysteine - set default values
        features['cys_first_pos'] = -1
        features['cys_middle_dist'] = 31  # Max distance
        for offset in [-2, -1, 1, 2]:
            features[f'cys_ctx_{offset}'] = 'X'
    
    return features

# Test
test_cys_features = get_cysteine_features("AAAAAAACDEFGHIKLMNPQRSTVWYAAAA")
print(f"✓ Cysteine features function defined")
print(f"  Example features: {dict(list(test_cys_features.items())[:4])}")

✓ Cysteine features function defined
  Example features: {'cys_count': 1, 'cys_present': 1, 'cys_first_pos': 7, 'cys_middle_dist': 8}


### 3.3 Physicochemical Properties

In [39]:
def get_physicochemical_features(sequence):
    """
    Calculate physicochemical properties.
    
    Returns aggregated statistics: mean, std, etc.
    """
    features = {}
    
    # Hydrophobicity statistics
    hydro_scores = [HYDROPHOBICITY.get(aa, 0) for aa in sequence]
    features['hydro_mean'] = np.mean(hydro_scores)
    features['hydro_std'] = np.std(hydro_scores)
    
    # Molecular weight
    mw_scores = [MOLECULAR_WEIGHT.get(aa, 0) for aa in sequence]
    features['mw_mean'] = np.mean(mw_scores)
    
    # Charge
    charge_scores = [CHARGE.get(aa, 0) for aa in sequence]
    features['charge_total'] = np.sum(charge_scores)
    
    return features

# Test
test_phys_features = get_physicochemical_features(test_seq)
print(f"✓ Physicochemical features function defined")
print(f"  Example: {test_phys_features}")

✓ Physicochemical features function defined
  Example: {'hydro_mean': np.float64(-0.29666666666666675), 'hydro_std': np.float64(3.0210906786935223), 'mw_mean': np.float64(134.41433333333333), 'charge_total': np.float64(-0.7999999999999998)}


### 3.4 Combined Feature Extraction

In [40]:
def extract_features_lightweight(sequence):
    """
    Extract all features from a sequence (lightweight mode).
    
    All features must have consistent columns across sequences!
    """
    features = {}
    
    # 1. Amino acid composition (20 features)
    features.update(get_aa_composition(sequence))
    
    # 2. Cysteine features (~8 features)
    features.update(get_cysteine_features(sequence))
    
    # 3. Physicochemical properties (4 features)
    features.update(get_physicochemical_features(sequence))
    
    # 4. Position-specific features (stored as string for encoding later)
    features['middle_aa'] = sequence[15]  # Middle position
    
    # 5. Complexity (1 feature)
    features['unique_aa'] = len(set(sequence))
    
    return features


def extract_features_minimal(sequence):
    """
    Extract minimal features (if lightweight mode fails).
    
    Only AA composition + cysteine count (21 features)
    """
    features = {}
    
    # AA composition
    counter = Counter(sequence)
    for aa in AMINO_ACIDS:
        features[f'aa_{aa}'] = counter.get(aa, 0) / 31
    
    # Cysteine count only
    features['cys_count'] = counter.get('C', 0)
    
    return features

# Test both modes
lightweight_features = extract_features_lightweight(test_seq)
minimal_features = extract_features_minimal(test_seq)

print(f"✓ Feature extraction functions defined")
print(f"  Lightweight mode: {len(lightweight_features)} features")
print(f"  Minimal mode: {len(minimal_features)} features")

✓ Feature extraction functions defined
  Lightweight mode: 34 features
  Minimal mode: 21 features


## 4. Process Dataset in Chunks

**Why chunks?** Processing 89K sequences at once uses too much memory. We process 5,000 at a time.

In [41]:
import os

# Check if input file exists
if not os.path.exists(INPUT_FILE):
    print(f"❌ ERROR: {INPUT_FILE} not found!")
    print(f"   Please check the file path in the configuration cell.")
else:
    print(f"✓ Input file found: {INPUT_FILE}")
    
    # Count total rows
    total_rows = sum(1 for _ in open(INPUT_FILE)) - 1  # Subtract header
    print(f"  Total sequences: {total_rows:,}")
    print(f"  Chunks to process: {(total_rows // CHUNK_SIZE) + 1}")

✓ Input file found: ../data/independent_test_input.csv
  Total sequences: 59,540
  Chunks to process: 12


In [42]:
print("="*80)
print("PROCESSING DATASET")
print("="*80)
print(f"\nMode: {MODE.upper()}")
print(f"Chunk size: {CHUNK_SIZE:,} sequences")
print(f"\nProcessing...\n")

# Select feature extraction function based on mode
if MODE == 'minimal':
    extract_func = extract_features_minimal
else:
    extract_func = extract_features_lightweight

first_chunk = True
chunk_num = 0

# Process in chunks
for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    chunk_num += 1
    print(f"[Chunk {chunk_num}] Processing {len(chunk):,} sequences...")
    
    # Extract features for this chunk
    features_list = []
    for idx, row in chunk.iterrows():
        seq_features = extract_func(row['Sequence'])
        features_list.append(seq_features)
    
    # Convert to DataFrame
    features_chunk = pd.DataFrame(features_list)
    
    # Combine with original columns
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    metadata_cols = ['ID', 'Sequence'] + label_cols
    
    chunk_with_features = pd.concat([
        chunk[metadata_cols].reset_index(drop=True),
        features_chunk.reset_index(drop=True)
    ], axis=1)
    
    # Save to CSV (append mode after first chunk)
    if first_chunk:
        chunk_with_features.to_csv(OUTPUT_FILE, mode='w', index=False, header=True)
        first_chunk = False
        print(f"  Created {OUTPUT_FILE}")
        print(f"  Features extracted: {len(features_chunk.columns):,}")
    else:
        chunk_with_features.to_csv(OUTPUT_FILE, mode='a', index=False, header=False)
    
    print(f"  Saved {len(chunk_with_features):,} rows")
    
    # Free memory
    del features_list, features_chunk, chunk_with_features
    gc.collect()

print("\n" + "="*80)
print("✓ PROCESSING COMPLETE!")
print("="*80)

PROCESSING DATASET

Mode: LIGHTWEIGHT
Chunk size: 5,000 sequences

Processing...

[Chunk 1] Processing 5,000 sequences...


KeyError: "['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation'] not in index"

## 5. Summary

Check the output file and feature statistics.

In [33]:
# Load first few rows to check
print("\nLoading output file for inspection...")
df_check = pd.read_csv(OUTPUT_FILE, nrows=5)

print(f"\nOutput file: {OUTPUT_FILE}")
print(f"Shape: {df_check.shape}")
print(f"\nColumns: {len(df_check.columns)}")
print(f"  - Metadata: 2 (ID, Sequence)")
print(f"  - Labels: 3")
print(f"  - Features: {len(df_check.columns) - 5}")

# File size
file_size = os.path.getsize(OUTPUT_FILE)
if file_size < 1024*1024:
    print(f"\nFile size: {file_size/1024:.1f} KB")
else:
    print(f"\nFile size: {file_size/(1024*1024):.1f} MB")

print("\nFirst few rows:")
df_check.head()


Loading output file for inspection...

Output file: ../data_engineered/train_augmented_v2_with_features.csv
Shape: (5, 39)

Columns: 39
  - Metadata: 2 (ID, Sequence)
  - Labels: 3
  - Features: 34

File size: 41.0 MB

First few rows:


,ID,Sequence,S-glutathionylation,S-nitrosylation,S-palmitoylation,aa_A,aa_C,aa_D,aa_E,aa_F,...,cys_ctx_-2,cys_ctx_-1,cys_ctx_1,cys_ctx_2,hydro_mean,hydro_std,mw_mean,charge_total,middle_aa,unique_aa
0,Q9H4H8,AAAAAAEDSFGSSHDCSSGTYFPEQSDLEPP,0.0,0.0,0.0,0.193548,0.032258,0.096774,0.096774,0.064516,...,H,D,S,S,-0.561290,2.243654,118.890968,-5.9,C,13
1,Q8N697,AAAAAAGAFAGRRAACGAVLLTELLERAAFY,0.0,1.0,0.0,0.419355,0.032258,0.000000,0.064516,0.064516,...,A,A,G,A,0.877419,2.562095,115.932581,1.0,C,10
2,Q8IP90,AAAAAAGKQIEGPEGCNLFIYHLPQEFTDTD,0.0,0.0,0.0,0.193548,0.032258,0.064516,0.096774,0.064516,...,E,G,N,L,-0.216129,2.788351,123.193871,-3.9,C,15
3,P36897,AAAAAALLPGATALQCFCHLCTKDNFTCVTD,0.0,0.0,0.0,0.258065,0.129032,0.064516,0.000000,0.064516,...,L,Q,F,C,0.758065,2.560969,118.775161,-0.9,C,13
4,Q8NCN4,AAAAAALSRRGRRGRCDETAAAKTGAPGPAS,0.0,0.0,0.0,0.354839,0.032258,0.032258,0.032258,0.000000,...,G,R,D,E,-0.487097,2.526982,113.183548,4.0,C,11


## 6. Next Steps

Now that features are extracted, you need to:

1. **Encode categorical features** → Run `03_encode_categorical_features.ipynb`
   - Converts amino acid letters to one-hot encoded columns
   - Creates `train_with_features_encoded.csv`

2. **Create train/val split** → Run `04_train_val_split.ipynb`

3. **Train models** → RF and XGBoost

### Feature Quality Check:
- ✓ All sequences processed
- ✓ Consistent feature columns
- ✓ No missing values
- ⚠️  Some features are categorical (will be encoded next)

### Memory Usage:
- Chunked processing: ~500 MB RAM
- Output file: ~20-30 MB